# 🛞⚛️ Quantum Vision Transformers for Tyre Defect Detection
## CNN vs. ViT vs. QCNN vs. QViT — a benchmark you can *see*

**Industrial context:** Apollo-Tyres-style automated quality control.  
**Dataset:** TyreNet (Mendeley) — ~1,700 images, *good* vs. *defective* tyres.  
**Stack:** PyTorch (classical) + PennyLane (variational quantum circuits) on `lightning.gpu`.

---
### The hypothesis we are going to *prove practically*
> Hybrid **Quantum Vision Transformers (QViTs)** match classical accuracy with **far fewer trainable parameters** ($O(n)$ vs $O(n^2)$ attention scaling) and develop a **global inductive bias** that makes them attend to the actual defect — not just local texture.

Grounded in **Cherrat et al. (2024)** (QSA parameter scaling) and **Boucher et al. (2025)** (inductive bias of quantum attention).

### How we prove it — four kinds of evidence
1. **Parameter-efficiency frontier** — accuracy vs. #params, with the Pareto frontier drawn.
2. **Explainability overlays** — Grad-CAM (CNN/QCNN) and attention-rollout (ViT/QViT) heatmaps laid *on the real tyre*, plus a quantitative **defect-focus score**.
3. **Representation quality** — t-SNE of each model's learned feature space.
4. **Quantum transparency** — we draw the circuits, plot qubit states on Bloch spheres, and chart circuit expressivity, so the 'quantum' is never a black box.

> 💡 *New to quantum?* Sections 3–4 are a self-contained mini-course on qubits, embeddings, and variational circuits, using the exact circuits this project trains.

## 0 · Environment setup
Enable a **GPU runtime** (*Runtime → Change runtime type → T4/A100*), then run the cell below to clone the repo and install PennyLane + `pennylane-lightning[gpu]`.

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/khushib004/QViT_Experiment.git'
BRANCH   = 'claude/benchmark-vision-defect-detection-n1erk'
REPO_DIR = '/content/QViT_Experiment' if IN_COLAB else os.path.abspath('..')

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR])
    subprocess.check_call(['pip', 'install', '-q', 'pennylane>=0.35',
                            'pennylane-lightning[gpu]', 'scikit-learn',
                            'matplotlib', 'tqdm', 'pillow', 'kagglehub'])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR, '| colab:', IN_COLAB)

In [ ]:
import torch, pennylane as qml, numpy as np, matplotlib.pyplot as plt

def pick_qdevice():
    for cand in ['lightning.gpu', 'lightning.qubit', 'default.qubit']:
        try:
            qml.device(cand, wires=2); return cand
        except Exception:
            continue
    return 'default.qubit'

QDEVICE = pick_qdevice()
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'PyTorch {torch.__version__} | PennyLane {qml.__version__}')
print(f'compute: {DEVICE} | quantum sim: {QDEVICE}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1 · Dataset — TyreNet from Kaggle (real images, no mocks)

We use the **[Tyre Quality Classification dataset](https://www.kaggle.com/datasets/warcoder/tyre-quality-classification)** (Kaggle, by `warcoder`) — ~1,854 real photos of `good` and `defective` tyres.

### Kaggle authentication (one-time)
Kaggle requires you to authenticate to download datasets. Pick one:

**(a) Google Colab — recommended:**
1. Open *Settings → Secrets* (the 🔑 icon in the left sidebar).
2. Add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (get them from your Kaggle account → *Settings → API → Create New Token*).
3. Toggle "Notebook access" on for both.

**(b) Desktop / VS Code:**
* Download `kaggle.json` from Kaggle, place it at `~/.kaggle/kaggle.json`, then `chmod 600 ~/.kaggle/kaggle.json`.
* Or export environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY` before running.

The next cell calls our helper script which handles secret propagation and folder normalisation automatically.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("warcoder/tyre-quality-classification")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
from scripts.download_data import download_tyrenet

DATA_ROOT = download_tyrenet('data/tyrenet')

# Sanity check
n_good = sum(1 for _ in (DATA_ROOT / 'good').rglob('*') if _.is_file())
n_def  = sum(1 for _ in (DATA_ROOT / 'defective').rglob('*') if _.is_file())
print(f'\nDataset: {n_good} good + {n_def} defective = {n_good + n_def} images')
assert n_good > 0 and n_def > 0, 'Dataset download failed — check Kaggle auth.'

In [ ]:
from src.data import TyreNetDataset
from src.explain import denormalize

N_QUBITS, PATCH_SIZE = 4, 32
kw = dict(root=str(DATA_ROOT), n_qubits=N_QUBITS, patch_size=PATCH_SIZE, reducer='conv')
train_ds = TyreNetDataset(split='train', **kw)
val_ds   = TyreNetDataset(split='val',   **kw)
test_ds  = TyreNetDataset(split='test',  **kw)
print(f'train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)} | quantum patches/img {train_ds.num_patches}')

fig, axes = plt.subplots(2, 5, figsize=(13, 5.4))
for ax, idx in zip(axes.flat, range(10)):
    s = train_ds[idx]
    ax.imshow(denormalize(s['image_classical']))
    ax.set_title('defective' if s['label'].item() else 'good', fontsize=10)
    ax.axis('off')
plt.suptitle('TyreNet samples (classical 224×224 view)'); plt.tight_layout(); plt.show()

### 1.1 · Dual-resolution: the quantum-ready view
Classical nets eat the full $3\times224\times224$ image. A quantum register cannot, so each image patch is compressed to `n_qubits` features (here a frozen strided-conv reducer, `tanh`-squashed to $[-1,1]$). Below: the per-patch feature matrix that gets angle-encoded into qubits.

In [ ]:
qv = train_ds[0]['image_quantum']  # (P, n_qubits)
fig, ax = plt.subplots(figsize=(7, 2.8))
im = ax.imshow(qv.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xlabel('patch index'); ax.set_ylabel('qubit feature')
ax.set_title(f'Quantum-ready features  {tuple(qv.shape)}  → input to AngleEmbedding')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 2 · A 5-minute quantum primer (using *our* circuits)

A **qubit** is a unit vector $\lvert\psi\rangle = \cos\tfrac{\theta}{2}\lvert0\rangle + e^{i\phi}\sin\tfrac{\theta}{2}\lvert1\rangle$ — a point on the **Bloch sphere**. Our circuit does three things:
1. **AngleEmbedding** rotates each qubit by $R_Y(x_i)$ → *loads classical data into quantum states*.
2. **BasicEntanglerLayers** applies $R_X(\theta)$ + a ring of CNOTs → *entangles qubits* (correlations no single qubit holds alone).
3. **Measurement** reads $\langle Z_i\rangle = P(0)-P(1)$ on each wire → back to classical numbers.

Let's *draw the actual circuit* the QViT uses.

In [ ]:
from src.explain import draw_circuit
print(draw_circuit(n_qubits=N_QUBITS, n_layers=2, ansatz='basic', reupload=True))

In [ ]:
# Watch state preparation move qubits on the Bloch sphere.
from src.explain import bloch_vectors, plot_bloch
x_in = np.array([0.9, -0.6, 0.3, -0.9])[:N_QUBITS]  # a sample feature vector
vecs = bloch_vectors(x_in * np.pi, n_qubits=N_QUBITS, n_layers=2)
plot_bloch(vecs, title='Qubit states after AngleEmbedding + entangling layer'); plt.show()

In [ ]:
# Expressivity: how much can the circuit's output vary as we deepen the ansatz?
# (Too shallow = underfits; too deep = barren plateaus. This guides our n_layers choice.)
from src.explain import expressivity_curve
expressivity_curve(n_qubits=N_QUBITS, max_layers=6); plt.show()

## 3 · Quantum Self-Attention vs. classical attention

Classical MHA learns three $D\times D$ matrices per head ($3D^2$ params, **$O(D^2)$**). Our QSA replaces each with a per-token VQC whose only weights scale as **$O(n_{qubits})$**. Same job, dramatically fewer parameters — let's measure it directly.

In [ ]:
import torch.nn as nn
from src.models import ClassicalMHA, HybridQuantumMultiHeadAttention

embed_dim, heads = 96, 2
cmha = ClassicalMHA(embed_dim, heads)
qmha = HybridQuantumMultiHeadAttention(embed_dim, n_heads=heads, n_qubits=N_QUBITS, n_layers=2, device_name=QDEVICE)
n = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
qn = sum(p.numel() for nm,p in qmha.named_parameters() if 'qlayer' in nm)
print(f'Classical MHA params : {n(cmha):>7,d}')
print(f'Quantum   MHA params : {n(qmha):>7,d}  (of which purely-quantum: {qn})')
print(f'Purely-quantum weights are the {qn}-param O(n) core — the rest is the classical bottleneck.')
tokens = torch.randn(2, 50, embed_dim)
print('QSA forward OK:', tuple(qmha(tokens).shape), '| cached attn:', tuple(qmha.last_attn.shape))

## 4 · Build the four architectures

In [ ]:
from src.models import ClassicalCNN, vit_tiny, QCNN, qvit_tiny

def fresh_models():
    return {
        'CNN' : ClassicalCNN(num_classes=2),
        'ViT' : vit_tiny(num_classes=2),
        'QCNN': QCNN(num_classes=2, n_qubits=N_QUBITS, n_layers=2, device_name=QDEVICE),
        'QViT': qvit_tiny(num_classes=2, n_qubits=N_QUBITS, n_heads=2, n_layers=2,
                          depth=4, embed_dim=96, patch_size=16, ansatz='basic',
                          reupload=True, device_name=QDEVICE),
    }

models = fresh_models()
for name, m in models.items():
    print(f'{name:>5s} : {sum(p.numel() for p in m.parameters() if p.requires_grad):>10,d} trainable params')

## 5 · Train
Quantum forward passes dominate runtime, so quantum models get a short **classical warm-up** (VQC frozen) before the expensive quantum params unfreeze. Bump `EPOCHS` up for real data.

In [ ]:
from torch.utils.data import DataLoader
from src.training import train_supervised, train_distilled, history_to_dict

EPOCHS = 20            # bump to 30+ once you've confirmed it runs end-to-end
BATCH  = 16
mk = lambda ds, s: DataLoader(ds, batch_size=BATCH, shuffle=s, num_workers=2, pin_memory=True)
train_loader, val_loader, test_loader = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)

hist = {}
hist['CNN'] = train_supervised(models['CNN'], train_loader, val_loader, epochs=EPOCHS, device=DEVICE, model_name='CNN')
hist['ViT'] = train_supervised(models['ViT'], train_loader, val_loader, epochs=EPOCHS, device=DEVICE, model_name='ViT')

In [ ]:
hist['QCNN'] = train_supervised(models['QCNN'], train_loader, val_loader, epochs=EPOCHS,
                                quantum_warmup=2, device=DEVICE, model_name='QCNN')

In [ ]:
# QViT student, distilled from the trained ViT teacher (KD stabilises VQC training).
hist['QViT'] = train_distilled(models['QViT'], models['ViT'], train_loader, val_loader,
                               epochs=EPOCHS, temperature=4.0, alpha=0.5, device=DEVICE, model_name='QViT')

In [ ]:
# ─── 5.1  Save checkpoints ─── run once all four models have finished training ────
# Checkpoints go to results/checkpoints/ locally; also backed-up to Google Drive
# if you are in Colab so the weights survive a session timeout.
import shutil
from pathlib import Path

CKPT_DIR = Path('results/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    ckpt_path = CKPT_DIR / f'{name.lower()}.pt'
    h = hist[name]
    h.best_state = None   # already loaded into model; omit from file to halve size
    torch.save({'model_state': model.state_dict(), 'run_history': h}, ckpt_path)
    print(f'  saved  {name:>4s}  {ckpt_path}  ({ckpt_path.stat().st_size/1e6:.1f} MB)')

# Backup to Google Drive when running in Colab
if IN_COLAB:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        drive_dir = Path('/content/drive/MyDrive/QViT_checkpoints')
        drive_dir.mkdir(parents=True, exist_ok=True)
        for ckpt in CKPT_DIR.glob('*.pt'):
            shutil.copy(ckpt, drive_dir / ckpt.name)
        print(f'  Drive backup → {drive_dir}')
    except Exception as _e:
        print(f'  [Drive backup skipped] {_e}')


### 5.2 · Resume from saved checkpoints *(skip §5 training)*

If you have already trained and saved the models, run the cell below instead of
the training cells above. Uncomment to activate.


In [ ]:
# Uncomment to reload from saved checkpoints instead of re-running §5 training
#
# CKPT_DIR = Path('results/checkpoints')
# # or from Google Drive:
# # CKPT_DIR = Path('/content/drive/MyDrive/QViT_checkpoints')
#
# models, hist = fresh_models(), {}
# for name, model in models.items():
#     ckpt = torch.load(CKPT_DIR / f'{name.lower()}.pt', map_location=DEVICE)
#     model.load_state_dict(ckpt['model_state'])
#     hist[name] = ckpt['run_history']     # RunHistory object — works with §6+ as-is
#     h = hist[name]
#     print(f'  loaded {name:>4s} | best_val_acc={h.best_val_acc:.3f} | epochs={len(h.history)}')
# print('All models restored. Proceed to §6.')


## 6 · Evidence #1 — the parameter-efficiency frontier
The headline proof. If QViT/QCNN sit **up-and-to-the-left**, the $O(n)$ efficiency claim holds empirically.

In [ ]:
from src.utils import (count_flops, collect_predictions, classification_report,
                       plot_confusion, plot_roc_pr, plot_pareto_frontier,
                       plot_acc_vs_epochs, plot_flops_vs_gates)
from src.models import estimate_gate_count

sample = next(iter(val_loader))['image_classical'][:1].to(DEVICE)
gate = {
  'QCNN': estimate_gate_count(N_QUBITS,2,(28//2)**2,1,'basic',False)['total_gates']//3,
  'QViT': estimate_gate_count(N_QUBITS,2,(224//16)**2+1,2,'basic',True)['total_gates']*4,
}
records, preds_store = [], {}
for name, m in models.items():
    probs, preds, labels = collect_predictions(m, test_loader, DEVICE)
    rep = classification_report(probs, preds, labels)
    preds_store[name] = (probs, preds, labels)
    records.append({'model_name': name, 'n_params': hist[name].n_params,
                    'best_val_acc': hist[name].best_val_acc, 'test_acc': rep['accuracy'],
                    'test_f1': rep['f1'], 'test_auc': rep['auc'],
                    'flops': count_flops(m.to(DEVICE), sample),
                    'quantum_gates': gate.get(name, 0), 'is_quantum': name.startswith('Q')})

print(f"{'model':>5s} | {'params':>9s} | {'val':>5s} | {'test':>5s} | {'f1':>5s} | {'auc':>5s}")
for r in records:
    print(f"{r['model_name']:>5s} | {r['n_params']:>9,d} | {r['best_val_acc']:.3f} | {r['test_acc']:.3f} | {r['test_f1']:.3f} | {r['test_auc']:.3f}")
plot_pareto_frontier(records); plt.show()

In [ ]:
plot_acc_vs_epochs([history_to_dict(hist[k]) for k in models]); plt.show()
plot_flops_vs_gates(records); plt.show()

## 7 · Evidence #2 — diagnostics (confusion + ROC/PR)
For QC, **recall on the *defective* class** is what protects customers from shipped defects.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(3.4*len(models), 3.2))
for ax, (name,(probs,preds,labels)) in zip(axes, preds_store.items()):
    cm = np.zeros((2,2), int)
    for p,l in zip(preds, labels): cm[l,p]+=1
    ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j,i,cm[i,j],ha='center',va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set_title(name); ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['good','def']); ax.set_yticklabels(['good','def'])
    ax.set_xlabel('pred'); ax.set_ylabel('true')
plt.suptitle('Confusion matrices (test)'); plt.tight_layout(); plt.show()

plot_roc_pr([{'name':n,'probs':p,'labels':l} for n,(p,_,l) in preds_store.items()]); plt.show()

## 8 · ⭐ Evidence #3 — *where does each model look?*

This is the most compelling part. We overlay each model's saliency on real tyre images:
* **CNN & QCNN** → **Grad-CAM** (gradient-weighted conv activations).
* **ViT & QViT** → **Gradient-weighted attention rollout** (inspired by [jacobgil/vit-explain](https://github.com/jacobgil/vit-explain) and Chefer et al., 2021).

### Why gradient rollout beats vanilla rollout

| Method | What it shows | Class-specific? |
|---|---|---|
| **Vanilla rollout** (Abnar & Zuidema, 2020) | Compounds attention matrices: "where does CLS look?" | No — same map for every class |
| **Gradient rollout** (Chefer et al., 2021) | Weights each head's attention by the *gradient of the predicted-class logit* flowing through it, clamps negatives, then rolls up | **Yes** — the map answers "where did the model look *to decide defective*" |

For defect detection, class-specific saliency is critical: we need to know that the model's "defective" decision is grounded in the actual flaw, not in background texture. Our implementation adapts the jacobgil approach to work on both `ClassicalMHA` and our custom `HybridQuantumMultiHeadAttention` heads.

**Visual hypothesis:** the QViT's global attention should localise to the actual flaw, while the CNN may fixate on local texture irrespective of the defect.

In [ ]:
# Side-by-side: vanilla rollout vs. gradient rollout on one defective image.
# This shows WHY gradient rollout is better for defect detection.
from src.explain import attention_rollout, gradient_attention_rollout, overlay_heatmap, denormalize

defective_idxs = [i for i in range(len(test_ds)) if test_ds[i]['label'].item()==1][:6]
item = test_ds[defective_idxs[0]]
x = item['image_classical'].unsqueeze(0).to(DEVICE)

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for row, (name, m) in enumerate([('ViT', models['ViT']), ('QViT', models['QViT'])]):
    axes[row, 0].imshow(denormalize(item['image_classical']))
    axes[row, 0].set_title(f'{name} — original', fontsize=11)

    sal_vanilla = attention_rollout(m, x, discard_ratio=0.9)
    axes[row, 1].imshow(overlay_heatmap(item['image_classical'], sal_vanilla))
    axes[row, 1].set_title(f'{name} — vanilla rollout\n(class-agnostic)', fontsize=11)

    try:
        sal_grad = gradient_attention_rollout(m, x, class_idx=1, discard_ratio=0.9)
    except Exception:
        sal_grad = sal_vanilla  # fallback if backward fails
    axes[row, 2].imshow(overlay_heatmap(item['image_classical'], sal_grad))
    axes[row, 2].set_title(f'{name} — gradient rollout\n(class="defective")', fontsize=11)

for ax in axes.flat:
    ax.axis('off')
fig.suptitle('Vanilla vs. gradient-weighted attention rollout\n'
             '(gradient rollout highlights where the model looks to decide "defective")',
             fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
from src.explain import model_saliency, compare_models_figure

# pick a few defective test images so the comparison is meaningful
for i in defective_idxs[:4]:
    item = test_ds[i]
    x = item['image_classical'].unsqueeze(0).to(DEVICE)
    sal = {}
    for name, m in models.items():
        try:
            sal[name] = model_saliency(m, x)
        except Exception as e:
            print(f'[warn] {name} saliency failed on idx {i}: {e}')
    compare_models_figure(item['image_classical'], sal,
                          title=f'Defective sample {i} — saliency overlay')
    plt.show()

## 9 · Evidence #4 — learned representation space (t-SNE)
If a model carves a clean *good* vs *defective* boundary in its feature space, classification is easy and robust. Each model's `.embed()` gives the penultimate representation.

In [ ]:
from src.explain import extract_embeddings, project_2d

fig, axes = plt.subplots(1, len(models), figsize=(4*len(models), 3.8))
for ax, (name, m) in zip(axes, models.items()):
    try:
        emb, lab = extract_embeddings(m, test_loader, DEVICE, max_samples=300)
        e2 = project_2d(emb)
        for cls, cname, c in [(0,'good','#2a9d8f'),(1,'defective','#e76f51')]:
            mk_ = lab==cls
            ax.scatter(e2[mk_,0], e2[mk_,1], s=14, alpha=0.7, c=c, label=cname, edgecolors='none')
        ax.set_title(f'{name} features'); ax.set_xticks([]); ax.set_yticks([])
    except Exception as e:
        ax.set_title(f'{name}: {e}')
axes[0].legend(); plt.suptitle('Learned feature space (t-SNE)'); plt.tight_layout(); plt.show()

## 10 · Bonus — the QCNN's *quantum* feature maps
The Quanvolutional layer outputs one feature map per qubit. These are literally the quantum circuit's view of the tyre — a rare chance to *see* what a quantum kernel extracts.

In [ ]:
item = test_ds[defective_idxs[0]]
_ = models['QCNN'](item['image_classical'].unsqueeze(0).to(DEVICE))
maps = models['QCNN'].quanv.last_quantum_maps[0].cpu().numpy()  # (n_qubits, H', W')
fig, axes = plt.subplots(1, N_QUBITS+1, figsize=(3*(N_QUBITS+1), 3))
axes[0].imshow(denormalize(item['image_classical'])); axes[0].set_title('input'); axes[0].axis('off')
for q in range(N_QUBITS):
    axes[q+1].imshow(maps[q], cmap='viridis'); axes[q+1].set_title(f'qubit {q} ⟨Z⟩'); axes[q+1].axis('off')
plt.suptitle('Quanvolutional feature maps (one per qubit)'); plt.tight_layout(); plt.show()

## 11 · Analysis & conclusions

Read the figures together:

1. **Parameter efficiency (§6).** On the Pareto frontier, the quantum hybrids should reach comparable accuracy with **orders of magnitude fewer parameters** than `vit_tiny` — the empirical signature of QSA's $O(n)$ scaling (Cherrat 2024). The pure-quantum weight count printed in §3 (a few dozen numbers) is doing real representational work.

2. **Defect localisation (§8).** The saliency overlays show *where* each model looks. If the QViT attends to the actual flaw region while the CNN fixates on local texture irrespective of the defect, that is direct evidence of the *global inductive bias* argued by Boucher (2025) — and exactly what a QC engineer wants (decisions grounded in the real flaw, not surface artefacts).

3. **Representations (§9).** Cleaner class separation in t-SNE = more robust, transferable features.

4. **Honesty checks.** `lightning.gpu` *simulates* the circuits — real superconducting hardware adds noise and decoherence not modelled here. The conv reducer is frozen; letting it learn end-to-end is a strong next ablation. The Kaggle dataset is ~1,854 images — small by modern CV standards, so cross-validation or repeated runs with seed sweeps make the numbers more trustworthy.

### Push further (great learning projects)
* Swap `ansatz='strong'` and toggle `reupload` to study expressivity vs. trainability.
* Sweep `n_qubits ∈ {4,6,8}` and plot the frontier shift.
* Make the strided-conv reducer trainable and co-optimise it with the VQC.
* Run 3 seeds per model and report mean ± std — much more credible than single runs.

### Reproduce everything as a script
```bash
# First, download the dataset (one-time)
python scripts/download_data.py

# Then run the full benchmark
python scripts/benchmark.py --data_root data/tyrenet --epochs 25 \
    --n_qubits 4 --n_layers 2 --n_heads 2 --qvit_depth 4 \
    --qdevice lightning.gpu --use_kd
```
Outputs land in `results/plots/` (pareto, saliency overlays, embeddings, ROC/PR) and `results/logs/benchmark.json`.

## References
1. Cherrat, El Amine et al. *Quantum Vision Transformers.* (2024).
2. Boucher, P. et al. *Inductive bias of quantum attention.* (2025).
3. Henderson, M. et al. *Quanvolutional Neural Networks.* arXiv:1904.04767 (2019).
4. Hinton, G. et al. *Distilling the Knowledge in a Neural Network.* (2015).
5. Selvaraju, R. et al. *Grad-CAM.* ICCV (2017).
6. Abnar & Zuidema. *Quantifying Attention Flow in Transformers.* ACL (2020).
7. Chefer, H. et al. *Transformer Interpretability Beyond Attention Visualization.* CVPR (2021).
8. Pérez-Salinas et al. *Data re-uploading for a universal quantum classifier.* Quantum (2020).
9. jacobgil/vit-explain — [github.com/jacobgil/vit-explain](https://github.com/jacobgil/vit-explain) (gradient rollout implementation reference).